# 005: Finding innovation-culture sentences with embeddings and Gemini Flash

**Hands-on Exercise 1 — Frontier LLMs and RAG**

This notebook demonstrates a simplified classroom version of **Li, Mai, Shen, Yang & Zhang (2026)**, focusing only on **innovation and adaptability culture**.

The exercise is designed to be read step by step. Each stage explains both the coding task and the measurement idea behind it: how we move from raw earnings-call text to a smaller set of sentences that may capture a culture construct.

## Background: Li et al. (2026) and this classroom simplification

Li et al. (2026) use a modern hybrid workflow to measure corporate culture from text:

1. **Keyword search** for explicit culture terms
2. **BERT-style semantic / classifier filtering** to find culture discussion without exact keywords
3. **Generative-AI filtering and extraction**
4. **RAG** when more context is needed

This classroom notebook implements a simplified **first step**:

1. Keyword retrieval
2. Sentence-transformer semantic retrieval
3. Gemini Flash filtering

### Definition: innovation and adaptability culture

Innovation and adaptability culture refers to shared **values, norms, practices, mindset, behavioral expectations, routines, or ways of working** that support innovation, creativity, technology adoption, entrepreneurship, adaptability, transformation, flexibility, agility, experimentation, disruption, resilience to change, openness to change, taking initiative, or new ways of working.

### Important distinction

A sentence is **NOT** innovation/adaptability culture merely because it mentions:

- a new product
- technology, AI, or R&D
- innovation spending or digital transformation
- market disruption or growth opportunities
- a new business strategy

It must discuss innovation/adaptability as a **culture, mindset, capability, way of working, organizational practice, leadership approach, routine, or behavioral pattern**.

**Pedagogical message:** Retrieval is part of measurement construction — not a neutral preprocessing step.

## Setup: packages, paths, and API access

This section prepares the notebook for GitHub and Colab use. The package installation cell makes the notebook easier to run on a new machine, while the path variables keep all output files in the current runtime.

The input CSV is loaded directly from the GitHub `raw` branch, so students do not need to upload the data manually in Colab.

The Gemini API key is read from Colab Secrets using `userdata.get("Gemini_API_Key")`. Outside Colab, the same code falls back to the local environment variable `Gemini_API_Key`. We do not hard-code API keys in notebooks because notebooks are often shared with students or uploaded to GitHub.

In [ ]:
# Quiet package installation (no GPU required)
import subprocess
import sys

packages = [
    "pandas", "numpy", "scikit-learn", "sentence-transformers",
    "tqdm", "google-genai", "json-repair", "matplotlib",
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q"] + packages
)

In [ ]:
import json
import os
import re
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from google import genai
from google.genai import types
from json_repair import repair_json
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    userdata = None
    IN_COLAB = False

GITHUB_RAW_BASE = (
    "https://raw.githubusercontent.com/"
    "helenlu-vbs/NLP_LLM_for_Finance_and-Accounting_Research-Sheffield-/raw"
)
INPUT_FILE = f"{GITHUB_RAW_BASE}/003_all_US_calls_2024Q4_top500.csv"

# In Colab, outputs are saved to /content by default. Locally, use the teaching folder.
PROJECT_DIR = Path("/content") if IN_COLAB else Path(r"C:\Users\Helen\Dropbox\Sheffield_NLP_teaching")
OUTPUT_DIR = PROJECT_DIR

MAX_SENTENCES = 5000
MAX_LLM_SENTENCES = 60  # Set MAX_LLM_SENTENCES = 20 for a quick classroom demo
BATCH_SIZE = 20
TEMPERATURE = 0
MODEL_NAME = "gemini-2.5-flash-lite"


def get_gemini_api_key():
    if userdata is not None:
        try:
            key = userdata.get("Gemini_API_Key")
            if key:
                return key
        except Exception:
            pass
    return os.getenv("Gemini_API_Key")


api_key = get_gemini_api_key()
client = genai.Client(api_key=api_key) if api_key else None

print(f"Running in Colab: {IN_COLAB}")
print(f"Input file: {INPUT_FILE}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Gemini_API_Key set: {bool(api_key)}")

## 1. Load earnings-call turns

We start with the raw unit available in the CSV: an earnings-call **turn**. A turn is one speaker contribution, not yet a sentence. Loading the data first lets us inspect the columns and confirm that the text field we will analyze is `turn_text`.

This step also removes empty turns, because missing text cannot contribute to either keyword search, embeddings, or LLM filtering.

In [ ]:
df = pd.read_csv(INPUT_FILE)
print("Shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head())

df["call_date"] = pd.to_datetime(df["call_date"], errors="coerce")
before = len(df)
df = df.dropna(subset=["turn_text"]).copy()
print(f"Dropped {before - len(df)} rows with missing turn_text. Remaining: {len(df)}")

## 2. Split turns into sentences

Most culture claims are easier to evaluate at the **sentence** level than at the full-turn level. A full answer may contain several topics, while a sentence usually gives a cleaner unit for retrieval and validation.

We keep the call and speaker metadata so that every sentence can still be traced back to the company, call date, speaker, and original transcript component. The length filter removes very short fragments and very long passages that are less useful for a classroom demonstration.

In [ ]:
METADATA_COLS = [
    "companyname", "companyid", "transcriptid", "call_date", "headline",
    "turn_index", "transcriptcomponentid", "transcriptcomponenttypeid",
    "component_type", "speaker_name", "speaker_type",
]

SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")


def split_into_sentences(text: str) -> list[str]:
    text = str(text).strip()
    if not text:
        return []
    parts = SENTENCE_SPLIT_RE.split(text)
    return [p.strip() for p in parts if p.strip()]


def word_count(s: str) -> int:
    return len(re.findall(r"\b\w+\b", s))


rows = []
sent_counter = 0
for _, row in tqdm(df.iterrows(), total=len(df), desc="Splitting turns"):
    for sentence in split_into_sentences(row["turn_text"]):
        sent_counter += 1
        rec = {col: row[col] for col in METADATA_COLS}
        rec["sent_id"] = sent_counter
        rec["sentence"] = sentence
        rows.append(rec)

sentences_df = pd.DataFrame(rows)
print(f"Raw sentences: {len(sentences_df):,}")

sentences_df = sentences_df.drop_duplicates(subset=["sentence"]).copy()
sentences_df["n_words"] = sentences_df["sentence"].map(word_count)
sentences_df = sentences_df[
    (sentences_df["n_words"] >= 8) & (sentences_df["n_words"] <= 120)
].copy()
print(f"After dedup + length filter (8–120 words): {len(sentences_df):,}")

if len(sentences_df) > MAX_SENTENCES:
    sentences_df = sentences_df.sample(n=MAX_SENTENCES, random_state=42).sort_values("sent_id")
    print(f"Sampled to MAX_SENTENCES={MAX_SENTENCES}")

sentences_df = sentences_df.reset_index(drop=True)
display(sentences_df.head())

## Stage 1: Keyword retrieval for innovation and adaptability culture

Keyword retrieval is the most transparent way to begin: we can see exactly which phrases caused a sentence to be selected. This is valuable for teaching because students can immediately audit the dictionary.

The trade-off is that keyword search is narrow. It finds explicit phrases such as “culture of innovation” or “ways of working,” but it may miss sentences that discuss the same idea using different language. We also avoid broad standalone words such as “innovation” or “AI” because they often refer to products or spending rather than organizational culture.

In [ ]:
INNOVATION_KEYWORD_PHRASES = [
    "innovation culture", "culture of innovation", "innovative culture",
    "innovation mindset", "innovative mindset", "entrepreneurial culture",
    "entrepreneurial mindset", "agile culture", "adaptive culture",
    "adaptability", "agility", "flexibility", "experimentation",
    "culture of experimentation", "willingness to experiment", "test and learn",
    "continuous improvement", "transformation mindset", "ways of working",
    "new ways of working", "resilience to change", "taking initiative",
    "initiative", "creativity", "creative culture", "disruptive mindset",
    "fast-moving organization", "fast moving organization", "break tradition",
    "openness to change", "change mindset",
]

# Phrase patterns: word boundaries where appropriate; case-insensitive
KEYWORD_PATTERNS = [
  (phrase, re.compile(r"\b" + re.escape(phrase) + r"\b", re.IGNORECASE))
  for phrase in INNOVATION_KEYWORD_PHRASES
]


def match_keywords(sentence: str) -> tuple[bool, str]:
    matched = [phrase for phrase, pat in KEYWORD_PATTERNS if pat.search(sentence)]
    return bool(matched), "; ".join(matched)


hits = sentences_df["sentence"].map(match_keywords)
sentences_df["innovation_keyword_hit"] = hits.map(lambda x: x[0])
sentences_df["innovation_matched_keywords"] = hits.map(lambda x: x[1])

n_hits = sentences_df["innovation_keyword_hit"].sum()
pct_hits = 100 * n_hits / len(sentences_df)
print(f"Keyword hits: {n_hits:,} ({pct_hits:.2f}% of sentences)")

print("\n--- 15 example keyword-hit sentences ---")
examples = sentences_df[sentences_df["innovation_keyword_hit"]].head(15)
for i, row in examples.iterrows():
    print(f"\n[{row['innovation_matched_keywords']}] {row['sentence'][:300]}")

**Reflection:** Which keyword hits are genuine innovation-culture discussion, and which are simply product, technology, or strategy discussion?

## Stage 2: Semantic retrieval

Semantic retrieval asks a different question from keyword search: not “does this sentence contain one of our phrases?” but “is this sentence close in meaning to our culture concept?”

We use a sentence-transformer model to turn each sentence and the culture query into vectors. Cosine similarity then ranks sentences by how close they are to the query. This helps surface implicit examples, where a company describes agile practices, experimentation, or adaptability without using the exact dictionary phrases.

In [ ]:
SEMANTIC_QUERY = (
    "organizational culture about innovation, creativity, technology adoption, "
    "entrepreneurship, adaptability, transformation, flexibility, agility, "
    "experimentation, openness to change, disruption, resilience to change, "
    "taking initiative, and new ways of working"
)

embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
sentence_list = sentences_df["sentence"].tolist()
sentence_embeddings = embed_model.encode(
    sentence_list, show_progress_bar=True, batch_size=64
)
query_embedding = embed_model.encode([SEMANTIC_QUERY])
sentences_df["innovation_semantic_score"] = cosine_similarity(
    query_embedding, sentence_embeddings
)[0]

top200_ids = set(
    sentences_df.nlargest(200, "innovation_semantic_score")["sent_id"].tolist()
)
semantic_mask = (
    sentences_df["sent_id"].isin(top200_ids)
    | (sentences_df["innovation_semantic_score"] >= 0.35)
)
candidate_mask = sentences_df["innovation_keyword_hit"] | semantic_mask

innovation_candidates_df = sentences_df[candidate_mask].copy()


def candidate_reason(row) -> str:
    kw = bool(row["innovation_keyword_hit"])
    sem = row["sent_id"] in top200_ids or row["innovation_semantic_score"] >= 0.35
    if kw and sem:
        return "both"
    if kw:
        return "keyword_only"
    return "semantic_only"


innovation_candidates_df["candidate_reason"] = innovation_candidates_df.apply(
    candidate_reason, axis=1
)

print("Candidate counts by reason:")
print(innovation_candidates_df["candidate_reason"].value_counts())


def show_examples(reason: str, n: int = 10):
    print(f"\n=== {reason} (n={n}) ===")
    subset = innovation_candidates_df[
        innovation_candidates_df["candidate_reason"] == reason
    ].head(n)
    for _, row in subset.iterrows():
        print(f"\nscore={row['innovation_semantic_score']:.3f} | {row['sentence'][:280]}")


for r in ["keyword_only", "semantic_only", "both"]:
    show_examples(r, 10)

reason_counts = innovation_candidates_df["candidate_reason"].value_counts()
plt.figure(figsize=(6, 4))
reason_counts.plot(kind="bar", color=["#4C72B0", "#DD8452", "#55A868"])
plt.title("Innovation culture candidates by retrieval reason")
plt.xlabel("candidate_reason")
plt.ylabel("Number of sentences")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Semantic-only examples are the key teaching moment:** they show what keyword search would have missed.

## Stage 3: Gemini Flash filtering

The retrieval steps intentionally cast a wide net. Some candidate sentences will be real culture discussion, but many will be false positives about products, technologies, strategy, or market change.

Gemini Flash is used here as a **filter and explainer**. It reads a small balanced sample of candidate sentences and decides whether each one truly discusses innovation and adaptability as organizational culture. The explanations are useful for classroom auditing: students can ask whether the model’s reason matches the definition.

In [ ]:
INNOVATION_CULTURE_FILTER_PROMPT = """
You are helping with an academic textual-analysis exercise inspired by Li, Mai, Shen, Yang & Zhang (2026) on corporate culture.

Task:
For each earnings-call sentence below, decide whether it substantively discusses innovation and adaptability as a form of organizational culture.

Definition:
Innovation and adaptability culture refers to shared values, norms, practices, mindset, behavioral expectations, routines, or ways of working that support innovation, creativity, technology adoption, entrepreneurship, adaptability, transformation, flexibility, agility, experimentation, disruption, resilience to change, openness to change, taking initiative, or new ways of working.

Important:
Do NOT classify a sentence as innovation/adaptability culture merely because it mentions:
- a new product
- technology
- AI
- software
- R&D
- innovation spending
- digital transformation
- market disruption
- growth opportunities
- a new business strategy

The sentence must connect these topics to the organization's culture, mindset, practices, behavior, leadership approach, routines, capabilities, or way of working.

Examples of culture-related discussion:
- "We have built a culture of experimentation."
- "Our teams are adopting more agile ways of working."
- "The organization has become more entrepreneurial and willing to take initiative."
- "Management is encouraging innovation across the company."

Examples that are NOT necessarily culture-related:
- "We launched a new AI product."
- "R&D spending increased this quarter."
- "Digital revenue grew by 20%."
- "The market is experiencing disruption."

Return strict JSON only.
Return a list with one object per input sentence.

Each object must have:
- sent_id
- is_innovation_adaptability_culture: true or false
- explicit_or_implicit: explicit, implicit, or not_culture
- confidence: number between 0 and 1
- evidence_phrase: a short exact phrase from the sentence, or "" if not culture-related
- reason: one concise sentence explaining the decision

Sentences:
{sentences_json}
"""


def balanced_gemini_sample(candidates: pd.DataFrame, max_n: int) -> pd.DataFrame:
    reasons = ["keyword_only", "semantic_only", "both"]
    per_group = max(1, max_n // len(reasons))
    parts = []
    for reason in reasons:
        subset = candidates[candidates["candidate_reason"] == reason]
        if len(subset) == 0:
            continue
        n_take = min(per_group, len(subset))
        parts.append(subset.sample(n=n_take, random_state=42))
    sample = pd.concat(parts).drop_duplicates(subset=["sent_id"])
    if len(sample) > max_n:
        sample = sample.sample(n=max_n, random_state=42)
    return sample.sort_values("sent_id").reset_index(drop=True)


def parse_gemini_json(text: str) -> list:
    text = text.strip()
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        parsed = json.loads(repair_json(text))
    if isinstance(parsed, dict) and "results" in parsed:
        parsed = parsed["results"]
    if not isinstance(parsed, list):
        raise ValueError(f"Expected JSON list, got {type(parsed)}")
    return parsed


def run_gemini_batch(batch_df: pd.DataFrame, max_retries: int = 3) -> list:
    payload = [
        {"sent_id": int(row.sent_id), "sentence": row.sentence}
        for row in batch_df.itertuples(index=False)
    ]
    prompt = INNOVATION_CULTURE_FILTER_PROMPT.format(
        sentences_json=json.dumps(payload, ensure_ascii=False)
    )

    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=TEMPERATURE,
                    response_mime_type="application/json",
                ),
            )
            return parse_gemini_json(response.text)
        except Exception as exc:
            last_error = exc
            wait_seconds = 10 * attempt
            print(
                f"Gemini call failed on attempt {attempt}/{max_retries}: {exc}. "
                f"Waiting {wait_seconds} seconds before retrying."
            )
            time.sleep(wait_seconds)

    raise RuntimeError(f"Gemini failed after {max_retries} attempts") from last_error


if client is None:
    raise ValueError(
        "Set Gemini_API_Key in Colab Secrets (left sidebar > key icon), "
        "or set a local environment variable named Gemini_API_Key before running Stage 3."
    )

gemini_input_df = balanced_gemini_sample(innovation_candidates_df, MAX_LLM_SENTENCES)
print(f"Gemini input sample: {len(gemini_input_df)} sentences")
print(gemini_input_df["candidate_reason"].value_counts())

gemini_results = []
for start in tqdm(range(0, len(gemini_input_df), BATCH_SIZE), desc="Gemini batches"):
    batch = gemini_input_df.iloc[start : start + BATCH_SIZE]
    gemini_results.extend(run_gemini_batch(batch))

gemini_df = pd.DataFrame(gemini_results)
gemini_df = gemini_df.rename(
    columns={
        "is_innovation_adaptability_culture": "gemini_is_innovation_adaptability_culture",
        "explicit_or_implicit": "gemini_explicit_or_implicit",
        "confidence": "gemini_confidence",
        "evidence_phrase": "gemini_evidence_phrase",
        "reason": "gemini_reason",
    }
)
gemini_df["sent_id"] = gemini_df["sent_id"].astype(int)

innovation_gemini_df = gemini_input_df.merge(gemini_df, on="sent_id", how="left")
display(innovation_gemini_df.head())

## Save outputs

Saving intermediate files is part of good research workflow. Each file corresponds to a stage in the measurement pipeline, so students can inspect what changed after sentence splitting, retrieval, and LLM filtering.

This also makes the exercise reproducible: if a later step looks wrong, we can go back to the previous CSV and diagnose where the issue entered.

In [ ]:
sentences_out = OUTPUT_DIR / "005_innovation_sentences.csv"
candidates_out = OUTPUT_DIR / "005_innovation_candidates.csv"
gemini_out = OUTPUT_DIR / "005_innovation_gemini_filtered.csv"
validation_out = OUTPUT_DIR / "005_innovation_manual_validation_sample.csv"

sentences_df.to_csv(sentences_out, index=False)
innovation_candidates_df.to_csv(candidates_out, index=False)
innovation_gemini_df.to_csv(gemini_out, index=False)

print(f"Saved: {sentences_out}")
print(f"Saved: {candidates_out}")
print(f"Saved: {gemini_out}")

## Manual validation sample

The Gemini labels are useful, but they are not automatically a research variable. A researcher still needs to inspect a validation sample and compare model labels with human judgment.

This sample is deliberately balanced across retrieval sources and Gemini outcomes. That makes it easier to see where the method performs well and where it fails, instead of only reviewing the most obvious cases.

In [ ]:
def build_manual_validation_sample(gemini_scored: pd.DataFrame, n: int = 30) -> pd.DataFrame:
    scored = gemini_scored.dropna(subset=["gemini_is_innovation_adaptability_culture"]).copy()
    if scored.empty:
        return pd.DataFrame()

    scored["gemini_is_innovation_adaptability_culture"] = scored[
        "gemini_is_innovation_adaptability_culture"
    ].map(lambda x: str(x).lower() in {"true", "1", "yes"})

    buckets = {
        "keyword_only": scored[scored["candidate_reason"] == "keyword_only"],
        "semantic_only": scored[scored["candidate_reason"] == "semantic_only"],
        "both": scored[scored["candidate_reason"] == "both"],
        "gemini_positive": scored[scored["gemini_is_innovation_adaptability_culture"]],
        "gemini_negative": scored[~scored["gemini_is_innovation_adaptability_culture"]],
    }
    per_bucket = max(1, n // len(buckets))
    picked = []
    for name, subset in buckets.items():
        if len(subset) == 0:
            continue
        take = subset.sample(n=min(per_bucket, len(subset)), random_state=42)
        take = take.copy()
        take["validation_bucket"] = name
        picked.append(take)
    sample = pd.concat(picked).drop_duplicates(subset=["sent_id"])
    if len(sample) > n:
        sample = sample.sample(n=n, random_state=42)

    cols = [
        "sent_id", "sentence", "companyname", "call_date", "headline",
        "speaker_name", "speaker_type", "component_type",
        "innovation_keyword_hit", "innovation_matched_keywords",
        "innovation_semantic_score", "candidate_reason",
        "gemini_is_innovation_adaptability_culture", "gemini_explicit_or_implicit",
        "gemini_confidence", "gemini_evidence_phrase", "gemini_reason",
    ]
    sample = sample[[c for c in cols if c in sample.columns]].copy()
    sample["human_label_blank"] = ""
    sample["notes_blank"] = ""
    return sample.sort_values("sent_id").reset_index(drop=True)


validation_df = build_manual_validation_sample(innovation_gemini_df, n=30)
validation_df.to_csv(validation_out, index=False)
print(f"Saved: {validation_out} ({len(validation_df)} rows)")
display(validation_df.head())

## Gemini results visualization

The final chart gives a quick sense of how many retrieved candidates survive the LLM filter. This is not a final accuracy estimate, but it helps students see how much filtering happens after retrieval.

The high-confidence positive examples are useful for discussion because they show the kind of language the model treats as clear evidence of innovation and adaptability culture.

In [ ]:
gemini_scored = innovation_gemini_df.dropna(
    subset=["gemini_is_innovation_adaptability_culture"]
).copy()
gemini_scored["gemini_positive"] = gemini_scored[
    "gemini_is_innovation_adaptability_culture"
].map(lambda x: str(x).lower() in {"true", "1", "yes"})

counts = gemini_scored["gemini_positive"].value_counts().rename(
    index={True: "true innovation/adaptability culture", False: "not culture"}
)
plt.figure(figsize=(6, 4))
counts.plot(kind="bar", color=["#55A868", "#C44E52"])
plt.title("Gemini classification counts")
plt.ylabel("Number of sentences")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

top_positive = (
    gemini_scored[gemini_scored["gemini_positive"]]
    .sort_values("gemini_confidence", ascending=False)
    .head(10)[["sent_id", "sentence", "gemini_confidence", "gemini_evidence_phrase", "gemini_reason"]]
)
print("Top 10 highest-confidence Gemini-positive sentences:")
display(top_positive)

## Class discussion: from retrieval to measurement

1. Which keyword hits are false positives?
2. Which semantic-only examples look genuinely culture-related?
3. Did semantic retrieval find examples that keyword search missed?
4. Where does Gemini make mistakes?
5. Are Gemini's explanations useful for auditing the output?
6. What manual validation would be required before using this as a research variable?
7. How is this classroom workflow similar to and different from Li et al. (2026)?

## What we learned

- Keyword search is transparent but narrow.
- Broad standalone words such as "innovation" create many false positives.
- Embeddings help find implicit innovation-culture discussion.
- Frontier LLMs can filter and explain candidate passages.
- Model outputs are not research variables until validated.
- This notebook demonstrates the first step in a modern textual-analysis workflow: **retrieving candidate text for construct measurement**.